In [13]:
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
from helpers import *
from implementations import *

In [15]:
import os
import zipfile

DATA_PATH = "data/"

zip_path = os.path.join(DATA_PATH, "dataset_to_release.zip")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(DATA_PATH)

print(os.listdir(DATA_PATH))

['dataset', 'dataset_to_release.zip']


In [16]:

x_train, x_test, y_train, train_ids, test_ids = load_csv_data("data/dataset")

print("x_train:", x_train.shape)
print("x_test:", x_test.shape)
print("y_train:", y_train.shape)

x_train: (328135, 321)
x_test: (109379, 321)
y_train: (328135,)


In [17]:
import numpy as np
np.set_printoptions(precision=3, suppress=True, linewidth=150)

print(x_train[:5, :10])   # 5 premières lignes, 10 premières colonnes
print(y_train[:10])
print(train_ids[:10])

[[5.300e+01 1.100e+01 1.116e+07 1.100e+01 1.600e+01 2.015e+03 1.100e+03 2.015e+09 2.015e+09       nan]
 [3.300e+01 1.200e+01 1.215e+07 1.200e+01 1.500e+01 2.015e+03 1.200e+03 2.015e+09 2.015e+09 1.000e+00]
 [2.000e+01 1.000e+01 1.020e+07 1.000e+01 2.000e+01 2.015e+03 1.100e+03 2.015e+09 2.015e+09 1.000e+00]
 [4.200e+01 6.000e+00 6.182e+06 6.000e+00 1.800e+01 2.015e+03 1.100e+03 2.015e+09 2.015e+09       nan]
 [2.400e+01 1.100e+01 1.106e+07 1.100e+01 6.000e+00 2.015e+03 1.100e+03 2.015e+09 2.015e+09 1.000e+00]]
[-1 -1 -1 -1 -1 -1  1  1 -1 -1]
[0 1 2 3 4 5 6 7 8 9]


In [18]:
import csv

with open("data/dataset/x_train.csv", "r") as f:
    header = next(csv.reader(f))

print(len(header))
print(header[:20])

322
['Id', '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'COLGHOUS', 'STATERES', 'CELLFON3', 'LADULT', 'NUMADULT', 'NUMMEN', 'NUMWOMEN', 'CTELNUM1']


In [19]:
import csv
import numpy as np

with open("data/dataset/x_train.csv", "r") as f: #ouvrir un fichier csv, le lire et récupérer l'entête
    header = next(csv.reader(f)) #recupere seule l'entête du fichier csv
names = header[1:]   # on enlève "Id" car on ne veut pas l'analyser
assert len(names) == x_train.shape[1]

rows = []
for j, name in enumerate(names):
    col = x_train[:, j]
    valid = col[~np.isnan(col)]
    n_unique = len(np.unique(valid))
    rows.append([
        j, name,
        round(np.isnan(col).mean(), 3),          # la part de cases vides. Python compte True comme 1 et False comme 0, 
        #donc la moyenne donne directement le pourcentage de vides. round(..., 3) arrondit à 3 chiffres après la virgule.
        n_unique,                                 # nombre de valeurs distinctes
        valid.min() if len(valid) else "",
        valid.max() if len(valid) else "",
        round(valid.mean(), 3) if len(valid) else "",#si la colonne a au moins une réponse, calcule ; sinon, laisse la case vide
    ])

with open("resume_colonnes.csv", "w", newline="") as f: #On crée un nouveau fichier resume_colonnes.csv. Le "w" veut dire qu’on l’ouvre pour écrire dedans.
    w = csv.writer(f)
    w.writerow(["index", "nom", "nan_ratio", "n_unique", "min", "max", "moyenne"])
    w.writerows(rows)

In [20]:
import os
print(os.path.abspath("resume_colonnes.csv"))

/Users/rafaela.dsp/Desktop/Machine_Learning_CS/project1/resume_colonnes.csv


In [21]:
nan_ratio = np.isnan(x_train).mean(axis=0)
n_unique = np.array([len(np.unique(c[~np.isnan(c)])) for c in x_train.T])

a_garder = (nan_ratio < 0.9) & (n_unique > 1)
print("Questions gardées :", a_garder.sum(), "sur", len(names))

Questions gardées : 218 sur 321


In [22]:
y01 = (y_train == 1) #on crée un tableau de booléens qui vaut True si la réponse est 1 et False sinon. Cela permet de calculer la corrélation entre chaque question et la réponse 1.
scores = []
for j in np.where(a_garder)[0]:#données les indices des colonnes à garder, on calcule la corrélation entre chaque colonne et la réponse 1. On remplace les valeurs manquantes par la moyenne de la colonne. 
    col = x_train[:, j].copy()
    col[np.isnan(col)] = np.nanmean(col)
    if col.std() > 0:#col.std() mesure à quel point les valeurs sont dispersées. Si toutes les valeurs sont identiques,
        # elle vaut 0, et le calcul suivant serait impossible (division par zéro). On saute alors cette question.
        scores.append((abs(np.corrcoef(col, y01)[0, 1]), names[j]))

for s, nom in sorted(scores, reverse=True)[:20]:
    print(f"{nom:<12} {s:.3f}")

GENHLTH      0.244
_AGE80       0.225
_AGEG5YR     0.219
EMPLOY1      0.207
_AGE_G       0.207
BPHIGH4      0.205
_HCVU651     0.196
_AGE65YR     0.185
_RFHLTH      0.172
_RFHYPE5     0.171
_DRDXAR1     0.168
DIABETE3     0.161
USEEQUIP     0.133
DIFFWALK     0.133
HAVARTH3     0.122
CHILDREN     0.114
QLACTLM2     0.114
TOLDHI2      0.111
INTERNET     0.108
_RFCHOL      0.108


In [23]:
for s, nom in sorted(scores)[:20]:   # les 20 scores les plus faibles, questions les moins corrélées avec la réponse 1, donc avec l'état malade
    print(f"{nom:<12} {s:.3f}")

RCSGENDR     0.000
NUMHHOL2     0.000
_VEG23       0.000
_PA30021     0.001
_FRTRESP     0.001
INCOME2      0.001
_MISVEGN     0.001
_FRUITEX     0.001
PREGNANT     0.001
_CPRACE      0.002
_CRACE1      0.002
_PASTRNG     0.002
FRUITJU1     0.002
PVTRESD1     0.002
CASTHDX2     0.002
_FRT16       0.002
_RAWRAKE     0.002
MAXDRNKS     0.002
_CLLCPWT     0.002
CAREGIV1     0.003


In [27]:
rng = np.random.default_rng(0)
idx = rng.choice(len(x_train), size=5000, replace=False)  # 5000 personnes au hasard
chosen_ones = x_train[idx]                                 # leurs réponses à toutes les questions

def corr(a, b):
    return np.abs(np.corrcoef(a, b)[0, 1])

quantite = chosen_ones.shape[1]   # nombre de QUESTIONS

for i in range(quantite):
    for j in range(i + 1, quantite):
        if corr(chosen_ones[:, i], chosen_ones[:, j]) > 0.95:
            print(f"{names[i]} et {names[j]} sont très similaires")

_STATE et _STSTR sont très similaires
IDATE et IMONTH sont très similaires


/Users/rafaela.dsp/miniforge3/envs/dataanalysis/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/rafaela.dsp/miniforge3/envs/dataanalysis/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


SEQNO et _PSU sont très similaires
EDUCA et _EDUCAG sont très similaires
_HCVU651 et _AGE65YR sont très similaires
_PRACE1 et _MRACE1 sont très similaires
_RACE et _RACEGR3 sont très similaires
_AGEG5YR et _AGE80 sont très similaires
_AGEG5YR et _AGE_G sont très similaires
_AGE80 et _AGE_G sont très similaires
_DRNKWEK et _RFDRHV5 sont très similaires
_MISFRTN et _FRTRESP sont très similaires
_MISFRTN et _FRUITEX sont très similaires
_FRTRESP et _FRTLT1 sont très similaires
_FRTRESP et _FRUITEX sont très similaires
_VEGRESP et _VEGLT1 sont très similaires
_VEGRESP et _VEGETEX sont très similaires
_FRTLT1 et _FRUITEX sont très similaires
_VEGLT1 et _VEGETEX sont très similaires
_TOTINDA et PAMISS1_ sont très similaires
MAXVO2_ et FC60_ sont très similaires
_PACAT1 et _PA150R2 sont très similaires
_PACAT1 et _PA300R2 sont très similaires
_PAINDX1 et _PA150R2 sont très similaires
_PAINDX1 et _PA300R2 sont très similaires
_PAINDX1 et _PA30021 sont très similaires
_PA150R2 et _PA300R2 sont 